## CESM-LE SWE Spatial Prediction

This notebook produces summary metrics for the spatial prediction/conditional simulation of SWE from fitted Bayesian transport map and nonstationary Gaussian process models. The notebooks assumes the [spatial prediction experiments](../examples/predict_blocks.md) have been performed and the results files are in the local directory. 

### Import Libraries

Import necessary libraries and define a summary function `qsummary`

In [1]:
import numpy
import numpy.ma as ma
from netCDF4 import Dataset
import pandas

def qsummary(df,vrlst):
    # Summarize with quantiles
    nmtch = df.shape[0] 
    dfout = pandas.DataFrame({'NSmp' : nmtch}, index=[0])
    for j in range(len(vrlst)):
        tmpdt = df[vrlst[j]]
        dtvld = tmpdt[numpy.isfinite(tmpdt)]
        dtvld = dtvld[dtvld != 0.0]
        vrnm = '%s_Mean' % (vrlst[j])
        dfout[vrnm] = numpy.mean(dtvld)

    return dfout


### Read Prediction Results

The CRPS and prediction interval results are stored in output NetCDF files.

In [2]:
mndt = 20050401
dtlst = [20050411, 20050412, 20050413, 20050414, 20050415, 20050416]
lclst = ['A', 'B', 'C', 'D', 'E', 'F']
nlc = len(lclst)

for j in range(nlc):

    # BTM scoring, training size = 24, index 2 in leading dim
    qfnm = 'LENS_NAmer_H2OSNO_PredBlockScoring_%d.nc' % (dtlst[j])
    ncclm = Dataset(qfnm,'r')
    crpsbtmn = ncclm.variables['btm_crps_nonlinear'][2,:,:,0]
    crpsbtmnblk = ncclm.variables['btm_crps_nonlinear_blkavg'][2,:,:]
    crpsbtml = ncclm.variables['btm_crps_linear'][2,:,:,0]
    crpsbtmlblk = ncclm.variables['btm_crps_linear_blkavg'][2,:,:]

    prdintn = ncclm.variables['btm_pred_quantile_nonlinear'][2,:,:,0,2] - ncclm.variables['btm_pred_quantile_nonlinear'][2,:,:,0,0]
    prdintnblk = ncclm.variables['btm_pred_quantile_nonlinear_blkavg'][2,:,:,2] - ncclm.variables['btm_pred_quantile_nonlinear_blkavg'][2,:,:,0]
    prdintl = ncclm.variables['btm_pred_quantile_linear'][2,:,:,0,2] - ncclm.variables['btm_pred_quantile_linear'][2,:,:,0,0]
    prdintlblk = ncclm.variables['btm_pred_quantile_linear_blkavg'][2,:,:,2] - ncclm.variables['btm_pred_quantile_linear_blkavg'][2,:,:,0]
    ncclm.close()

    gfnm = 'LENS_NAmer_H2OSNO_PredBlockScoring_GP_%d.nc' % (dtlst[j])
    ncclm = Dataset(gfnm,'r')
    crpsgp = ncclm.variables['gp_crps_nonstn'][2,:,:,0]
    crpsgpblk = ncclm.variables['gp_crps_nonstn_blkavg'][2,:,:]

    prdintg = ncclm.variables['gp_pred_quantile_nonstn'][2,:,:,0,2] - ncclm.variables['gp_pred_quantile_nonstn'][2,:,:,0,0]
    prdintgblk = ncclm.variables['gp_pred_quantile_nonstn_blkavg'][2,:,:,2] - ncclm.variables['gp_pred_quantile_nonstn_blkavg'][2,:,:,0]
    ncclm.close()

    scrfrm = pandas.DataFrame({'Location': lclst[j], 'CRPS_NBTM_Loc': ma.median(crpsbtmn,axis=None), \
                               'CRPS_LBTM_Loc': ma.median(crpsbtml,axis=None), 'CRPS_GP_Loc': ma.median(crpsgp,axis=None), 
                               'CRPS_NBTM_Blk': ma.median(crpsbtmnblk,axis=None), \
                               'CRPS_LBTM_Blk': ma.median(crpsbtmlblk,axis=None), 'CRPS_GP_Blk': ma.median(crpsgpblk,axis=None)}, index=[0])
    prdfrm = pandas.DataFrame({'Location': lclst[j], 'PIWidth_NBTM_Loc': ma.median(prdintn,axis=None), \
                               'PIWidth_LBTM_Loc': ma.median(prdintl,axis=None), 'PIWidth_GP_Loc': ma.median(prdintg,axis=None), 
                               'PIWidth_NBTM_Blk': ma.median(prdintnblk,axis=None), \
                               'PIWidth_LBTM_Blk': ma.median(prdintlblk,axis=None), 'PIWidth_GP_Blk': ma.median(prdintgblk,axis=None)}, index=[0])

    if j == 0:
        scr_all = scrfrm
        prd_all = prdfrm
    else:
        scr_all = pandas.concat([scr_all,scrfrm], ignore_index=True)
        prd_all = pandas.concat([prd_all,prdfrm], ignore_index=True)


In [3]:
scr_all

,Location,CRPS_NBTM_Loc,CRPS_LBTM_Loc,CRPS_GP_Loc,CRPS_NBTM_Blk,CRPS_LBTM_Blk,CRPS_GP_Blk
0,A,14.250569,13.819408,13.831612,4.142464,4.276845,5.428392
1,B,32.378967,32.975082,38.270287,4.697236,4.754094,7.373266
2,C,13.906114,14.444599,14.067510,8.539595,8.802061,12.030961
3,D,8.038168,8.081025,8.185616,2.881000,3.109364,3.718717
4,E,3.735660,3.889093,6.124095,1.217355,1.183064,1.570551
5,F,9.173318,9.568630,9.538393,4.852064,5.026497,4.848341


In [4]:
prd_all

,Location,PIWidth_NBTM_Loc,PIWidth_LBTM_Loc,PIWidth_GP_Loc,PIWidth_NBTM_Blk,PIWidth_LBTM_Blk,PIWidth_GP_Blk
0,A,108.170456,104.241486,126.247658,32.682842,31.966051,50.763466
1,B,290.126648,293.682343,323.532349,41.834526,42.216949,61.315216
2,C,122.898148,127.598923,100.460274,72.996643,73.661407,74.107773
3,D,73.611458,73.510612,85.865211,23.081543,22.595329,32.993935
4,E,102.494064,100.571533,213.205597,18.819740,18.347073,25.560061
5,F,87.048752,87.819290,80.530296,34.909073,35.292351,44.251778


### Read Log Score Results

The CRPS and prediction interval results are stored in output NetCDF files.

In [5]:
for j in range(nlc):

    # BTM scoring, training size = 24, index 2 in leading dim
    # Read scoring results
    #btmscrfl = 'LENS_NAmer_H2OSNO_PredBlockScoring_%d.nc' % (dtlst[j]) # ,LinearScore,LinearScoreCond
    btmscrfl = 'LENS_NAmer_H2OSNO_%d_BTMScoring_PredBlock.csv' % (dtlst[j])
    btmfrm = pandas.read_csv(btmscrfl, \
                        dtype = {'Epoch':int, 'TrainSize':int, \
                                 'NonLinScore':float, 'NonLinScoreCond': float, \
                                 'LinearScore':float, 'LinearScoreCond': float}, encoding='utf-8-sig')
    grpbtm = btmfrm.groupby(['TrainSize'])
    btmqs = grpbtm.apply(qsummary,vrlst=['NonLinScoreCond','LinearScoreCond'],include_groups=False)
    btmqs.reset_index(drop=False,inplace=True)
    btmqs.drop(['level_1'],axis=1,inplace=True)
    btmqs.rename(columns={'NSmp':'NSmpBTM'})

    gpscrfl = 'LENS_NAmer_H2OSNO_%d_GPScoring_PredBlock.csv' % (dtlst[j])
    gpfrm = pandas.read_csv(gpscrfl, \
                        dtype = {'Epoch':int, 'TrainSize':int, \
                                 'NonStatScore':float, 'NonStatScoreCond': float}, encoding='utf-8-sig')
    grpgp = gpfrm.groupby(['TrainSize'])
    gpqs = grpgp.apply(qsummary,vrlst=['NonStatScoreCond'],include_groups=False)
    gpqs.reset_index(drop=False,inplace=True)
    gpqs.drop(['level_1'],axis=1,inplace=True)
    gpqs.rename(columns={'NSmp':'NSmpGP'})

    scrmrg = pandas.merge(btmqs,gpqs,how='inner',on=['TrainSize'])
    scrmrg['Location'] = lclst[j]

    if j == 0:
        scrfrmfl = scrmrg
    else:
        scrfrmfl = pandas.concat([scrfrmfl, scrmrg], ignore_index=True)

# Subset
scrfrm24 = scrfrmfl[ scrfrmfl['TrainSize'] == 24]


In [6]:
scrfrm24

,TrainSize,NSmp_x,NonLinScoreCond_Mean,LinearScoreCond_Mean,NSmp_y,NonStatScoreCond_Mean,Location
2,24,50,-17.637858,-17.688142,50,-21.789171,A
6,24,50,-34.768061,-35.195657,50,-36.783320,B
10,24,50,-32.795223,-33.303720,50,-35.811975,C
14,24,50,-21.114055,-21.777743,50,-25.025590,D
18,24,50,-15.813517,-15.909430,50,-15.410942,E
22,24,50,-20.444221,-20.707418,50,-23.289034,F
